# JetRacer FA2026 - Speed Track (Road-Only Pure Pursuit) + Tune + Lai tay

Quy trinh chay Speed Track sau khi `git pull` tren Jetson Nano:

1. **Buoc 0 (KHONG CAN CHAY cho `pp_fast`)**: file `models/lane_road_only_fast.engine` (FP16 cho Jetson Nano JetPack 4.5.1 / TRT 7.1.3) da co san trong repo -> pull ve la chay ngay! Neu xe khac ban TensorRT, Buoc 1 se tu dong build lai tu `.onnx`.
2. **Buoc 1**: giu mac dinh `MODEL_MODE = 'pp_fast'` (Road-Only Pure Pursuit + model 1 kenh `Ch0` ~109 FPS Nano). Neu san choi den manh, co the doi sang `'pp_resse'`.
3. **Buoc 2**: chay cell **MO GIAO DIEN SPEED TRACK** -> bam **1. MO CAMERA**.
4. Dat xe vao lan duong, doi banner `MAT DUONG OK - BAM CHAY` roi bam **2. CHAY - BAM LINE**.
5. Keo cac thanh ben phai de dieu chinh **TOC DO thang**, **TOC DO trong cua**, **GOC LAI TOI DA** (va cac thanh cua UI cu khi chuyen ve che do cu).

**Truoc moi lan Run All**: `Kernel > Restart & Clear Output`. Hai notebook mo camera cung luc se bao `Failed to create CaptureSession`.

**AN TOAN**

- Mo len la trang thai DUNG. Khong lenh nao xuong phan cung cho den khi bam **2. CHAY - BAM LINE** hoac **LAI TAY**.
- Option B can vung mat duong xanh noi den xe qua 5 frame lien tiep de bam **CHAY**. Ga tang khi co huong lai.
- **DUNG KHAN CAP** cat ga ngay. Camera dung hoac loi -> tu dong cat ga.
- Lan dau test tren xe moi: **KE BANH KHOI MAT DAT**.

## 0. (Tuy chon) Build TensorRT FP16 engine — DA CO SAN cho `pp_fast`, bo qua duoc

In [ ]:
import os
if not os.path.exists('configs/default.yaml') and os.path.isdir('/home/jetson/JetsonRacer'):
    os.chdir('/home/jetson/JetsonRacer')
# models/lane_road_only_fast.engine da duoc dong goi san trong repo -> KHONG can chay lai!
# (Neu xe khac phien ban JetPack/TRT, Cell 1 se tu dong build lai tu .onnx).
# Chi bo dau `#` ben duoi khi ban muon ep build lai (--force) hoac build them model khac:
# !python3 tools/setup_jetson_models.py --model road_fast
print('Engine san sang:', os.path.exists('models/lane_road_only_fast.engine'))

## 1. Chon model & Kiem tra thu muc, engine

In [ ]:
import os
import socket
if not os.path.exists('configs/default.yaml') and os.path.isdir('/home/jetson/JetsonRacer'):
    os.chdir('/home/jetson/JetsonRacer')

# ===================== CHON MODEL BAM VACH =====================
# 'pp_fast'  -> KHUYEN DUNG: Road-Only Pure Pursuit + LaneTiny-Fast (Chi Ch0, 30+ FPS, giu default.yaml)
# 'pp_resse' -> DU PHONG:    Road-Only Pure Pursuit + LaneTiny-ResSE (Chi Ch0, 25-30 FPS, chong choi tot)
# 'fast'     -> Ban cu (Ch0+Ch2 CornerController): LaneTiny-Fast
# 'resse'    -> Ban cu (Ch0+Ch2 CornerController): LaneTiny-ResSE
# 'fastscnn' -> Ban cu (Ch0+Ch2 CornerController): FastSCNN-Lane
# 'cv'       -> CV co dien (khong dung model)
MODEL_MODE = 'pp_fast'
# ===============================================================

MODE_MAP = {
    'pp_fast':  ('configs/road_pure_pursuit.yaml',       'models/lane_road_only_fast.engine'),
    'pp_resse': ('configs/road_pure_pursuit_resse.yaml', 'models/lane_road_only_resse.engine'),
    'fast':     ('configs/cnn_option_b_fast.yaml',       'models/lane_option_b_fast.engine'),
    'resse':    ('configs/cnn_option_b.yaml',            'models/lane_option_b_resse.engine'),
    'fastscnn': ('configs/cnn_option_b_fastscnn.yaml',   'models/lane_option_b_fastscnn.engine'),
    'cv':       (None, None),
}
if MODEL_MODE not in MODE_MAP:
    raise ValueError('MODEL_MODE phai la pp_fast | pp_resse | fast | resse | fastscnn | cv')

CFG_OVERRIDE, ENGINE_PATH = MODE_MAP[MODEL_MODE]
OVERRIDES = [CFG_OVERRIDE] if CFG_OVERRIDE else []
DUNG_CNN = (CFG_OVERRIDE is not None)

required = [
    'tools/tune_lane_jupyter.py',
    'src/jetracer_baseline/tuning_ui.py',
    'src/jetracer_baseline/perception/lane.py',
    'src/jetracer_baseline/perception/shading.py',
    'configs/default.yaml',
]
if DUNG_CNN:
    required += [
        'src/jetracer_baseline/perception/lane_cnn.py',
    ] + OVERRIDES
    # Neu chua co .engine nhung co .onnx, TensorRTEngine se TU DONG build FP16 khi nap!
    onnx_path = os.path.splitext(ENGINE_PATH)[0] + '.onnx'
    if not os.path.exists(ENGINE_PATH) and not os.path.exists(onnx_path):
        required.append(ENGINE_PATH)

missing = [p for p in required if not os.path.exists(p)]
print('Hostname:', socket.gethostname())
print('Thu muc:', os.getcwd())
print('Che do:', MODEL_MODE, '->', OVERRIDES)
print('Files:', 'OK' if not missing else 'THIEU ' + ', '.join(missing))
if missing:
    if ENGINE_PATH and ENGINE_PATH in missing:
        print()
        print('Chua co file engine. Chay cell 0 o tren hoac go trong Terminal:')
        print('  python3 tools/setup_jetson_models.py')
    raise IOError('Thieu file, xem danh sach o tren')

print('Shading da hieu chuan:', os.path.exists('configs/shading.yaml'))

if DUNG_CNN:
    import sys
    if 'src' not in sys.path:
        sys.path.insert(0, 'src')
    from jetracer_baseline.config import load_config
    from jetracer_baseline.perception import build_lane_detector
    _det = build_lane_detector(load_config('configs/default.yaml', OVERRIDES))
    print('Engine nap OK:', type(_det).__name__, '|', ENGINE_PATH)
    if hasattr(_det, 'close'):
        _det.close()
    del _det

## 2. Nap ma nguon giao dien

In [ ]:
import importlib
import tools.tune_lane_jupyter as T
importlib.reload(T)
launch = T.launch
print('Da nap code. Keo xuong cell MO GIAO DIEN SPEED TRACK de chay.')

## 3. MO GIAO DIEN SPEED TRACK  <-- BAM O CELL NGAY DUOI

Cac nut chinh:

| Nut | Tac dung |
|---|---|
| **1. MO CAMERA** | bat camera, hien mat duong xanh Ch0 + truc tam BEV |
| **2. CHAY - BAM LINE** | bam khi hien `MAT DUONG OK - BAM CHAY` |
| **DUNG** | dung binh thuong & chot file log CSV |
| **DUNG KHAN CAP** | cat ga ngay lap tuc |
| Cac slider ben phai | TOC DO thang, TOC DO trong cua, CUA SOM, LAI CUA GAT, NHA LAI RA CUA, TRE VAO CUA, NHAY VUNG CUA, GIAM TOC TRUOC CUA, GOC LAI TOI DA |

Kiem tra bang XANH LA `MODEL CNN (TRT) - models/lane_road_only_fast.engine` truoc khi bam.

**`driver_kind='nvidia'` = XE CHAY THAT.** Lan dau: KE BANH KHOI MAT DAT.
Muon chi xem detect ma banh khong quay thi doi thanh `'dryrun'`.

In [ ]:
try:
    ui.close()
except NameError:
    pass

from tools.tune_lane_jupyter import launch

try:
    OVERRIDES
except NameError:
    OVERRIDES = ['configs/road_pure_pursuit.yaml']
    print('Chua chay cell 1 -> tu dat OVERRIDES =', OVERRIDES)

# simple=True      -> giao dien Speed Track voi camera, nut CHAY va cac thanh dieu chinh
# simple=False     -> giao dien day du (them tab Tay cam + Thu data + Luu config)
# driver_kind      -> 'nvidia' XE CHAY THAT | 'dryrun' banh khong quay
ui = launch(driver_kind='nvidia', overrides=OVERRIDES, simple=True)

## 4. (Tuy chon) Chay Speed Track bang CLI khong can UI

Mac dinh duoc comment de khi ban bam **Run All**, notebook KHONG tu chay xe ngay va KHONG xung dot camera voi giao dien o tren. Muon dung CLI thi bo dau `#` va dong UI truoc.

In [ ]:
# !python3 -m src.jetracer_baseline.cli run --task speed --driver nvidia --override configs/road_pure_pursuit.yaml --max-seconds 60 --record

## 5. Dong giao dien khi chay xong (chay rieng cell nay, KHONG de chay trong Run All)

In [ ]:
# Bo dau # va chay rieng cell nay khi muon DONG camera + giao dien:
# ui.close()